# 02 — Inyección de Anomalías Sintéticas

Inyectamos 6 tipos de anomalías para crear el ground truth de entrenamiento.

**Diferencia clave respecto a v3:** los rangos 'Fuera de Rango' se calculan con percentiles estadísticos (P0.1/P99.9) del propio dataset, no con límites físicos del hardware.

**Entrada:** `data/interim/01_datos_cargados.parquet`  
**Salida:** `data/interim/02_datos_inyectados.parquet`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos del paso anterior

In [ ]:
df_trabajo = pd.read_parquet(PARQUET_01)
df_original = df_trabajo.copy()
print(f"Dataset cargado: {df_trabajo.shape}")
print(f"Período: {df_trabajo['Fecha'].min()} → {df_trabajo['Fecha'].max()}")


## 1. Columnas objetivo
v2 incluye UVENT_cen y UVENT_lN (a diferencia de v3 donde se excluyeron por señal no fiable).

In [ ]:
columnas_sensores_y_actuadores = [
    'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
    'XCO2I', 'XHINV', 'XTINV', 'XTS',
    'UVENT_cen', 'UVENT_lN'
]

# Filtrar la lista para asegurar que solo contenga columnas existentes en el DataFrame df_trabajo
columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]

print(f"Se utilizarán las siguientes columnas para potencialmente inyectar NaNs: \n{columnas_objetivo_existentes}")
columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]
std_devs_originales = {col: df_original[col].std() for col in columnas_objetivo_existentes if col in df_original.columns}
print(f"Columnas objetivo ({len(columnas_objetivo_existentes)}): {columnas_objetivo_existentes}")


## 2. Datos Faltantes
Inyectamos NaNs en el 2% de las filas normales, afectando 1 sensor aleatorio por fila.

In [ ]:
def inyectar_datos_faltantes(df, porcentaje_filas_afectadas, columnas_objetivo, num_valores_nan_por_fila=1, random_state=42):
    """Inyecta NaNs vectorizado: selecciona filas y columnas de golpe."""
    if not columnas_objetivo:
        print('No se proporcionaron columnas objetivo.')
        return df
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    idx_normales = df_modificado.index[df_modificado['etiqueta_deteccion'] == 'normal'].to_numpy()
    n = int(len(idx_normales) * porcentaje_filas_afectadas)
    if n == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado
    filas = rng.choice(idx_normales, size=n, replace=False)
    print('Inyectando datos faltantes en ' + str(n) + ' filas...')
    cols = np.array(columnas_objetivo)
    num_cols = min(num_valores_nan_por_fila, len(cols))
    col_indices = np.array([df_modificado.columns.get_loc(c) for c in cols])
    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    for fila_idx in filas:
        elegidas = rng.choice(col_indices, size=num_cols, replace=False)
        for ci in elegidas:
            df_modificado.iat[fila_idx, ci] = np.nan
        df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx] = 'Datos Faltantes'
    print('Inyeccion de datos faltantes completada. ' + str(n) + ' filas modificadas.')
    return df_modificado

In [ ]:
if 'df_trabajo' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    porcentaje_filas_nan = 0.02  # Inyectar NaNs en el 2% de las filas 'normales'
    num_sensores_por_fila_afectada = 1 # En cada fila afectada, poner NaN en 1 valor aleatorio (sensor o actuador)

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    etiquetas_deteccion_antes = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    df_trabajo = inyectar_datos_faltantes(df_trabajo, 
                                            porcentaje_filas_nan, 
                                            columnas_objetivo_existentes, 
                                            num_sensores_por_fila_afectada)

    print("\n--- Resultados de la Inyección de 'Datos Faltantes' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))

    # Verificar cuántos NaNs se crearon en total en las columnas objetivo
    total_nans_inyectados_verificacion = 0
    for col in columnas_objetivo_existentes:
        total_nans_inyectados_verificacion += df_trabajo[col].isnull().sum()

else:
    print("Error: Asegúrate de que 'df_trabajo' esté definido y 'columnas_objetivo_existentes' esté correctamente populada.")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())

## 3. Sensor Atascado
Secuencias de valor constante de 5 a 20 minutos consecutivos.

In [ ]:
if 'df_trabajo' in locals():
    # Asegurar que el DataFrame esté ordenado por fecha
    if 'Fecha' in df_trabajo.columns:
        print("Asegurando que el DataFrame esté ordenado por 'Fecha'...")
        df_trabajo = df_trabajo.sort_values(by='Fecha').reset_index(drop=True)
        print("DataFrame ordenado por 'Fecha' y el índice ha sido reseteado.")
    else:
        print("Advertencia: La columna 'Fecha' no se encontró. No se puede asegurar el orden temporal para 'Sensor Atascado'.")

else:
    print("Error: 'df_trabajo' no está definido.")

In [ ]:
def inyectar_sensor_atascado(df, num_secuencias_stuck, duracion_min_stuck, duracion_max_stuck, columnas_objetivo, random_state=42):
    """
    Inyecta anomalias de Sensor Atascado en el DataFrame.

    Versión optimizada:
    - Comprueba que TODA la ventana tenga valores validos (no solo el primero)
    - Bucle while: sigue hasta crear num_secuencias_stuck secuencias (max 10x intentos)
    - Busqueda vectorizada con numpy cumsum: O(N) por intento
    - Semilla fija para reproducibilidad
    """
    if not columnas_objetivo:
        print('No se proporcionaron columnas objetivo.')
        return df

    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    es_normal = (df_modificado['etiqueta_deteccion'] == 'normal').values.copy()

    secuencias_creadas = 0
    n = len(df_modificado)
    max_intentos = num_secuencias_stuck * 10

    print('Intentando crear ' + str(num_secuencias_stuck) + ' secuencias de Sensor Atascado...')

    intentos = 0
    while secuencias_creadas < num_secuencias_stuck and intentos < max_intentos:
        intentos += 1
        columna = str(rng.choice(columnas_objetivo))
        duracion = int(rng.integers(duracion_min_stuck, duracion_max_stuck + 1))
        valores_col = df_modificado[columna].values
        max_start = n - duracion
        if max_start <= 0:
            continue

        # Mascara combinada: fila normal Y valor del sensor no es NaN
        # Usamos cumsum sobre esta mascara para garantizar que TODA la ventana es valida
        es_valido = es_normal & ~np.isnan(valores_col)
        cum = np.cumsum(es_valido.astype(np.int32))
        cum = np.concatenate([[0], cum])
        sumas_ventana = cum[duracion: max_start + duracion] - cum[:max_start]

        candidatos = np.where(sumas_ventana == duracion)[0]

        if len(candidatos) == 0:
            continue

        start_idx = int(rng.choice(candidatos))
        stuck_value = float(valores_col[start_idx])

        col_idx  = df_modificado.columns.get_loc(columna)
        det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
        tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')

        df_modificado.iloc[start_idx:start_idx + duracion, col_idx]  = stuck_value
        df_modificado.iloc[start_idx:start_idx + duracion, det_idx]  = 'anomalia'
        df_modificado.iloc[start_idx:start_idx + duracion, tipo_idx] = 'Sensor Atascado'
        es_normal[start_idx:start_idx + duracion] = False

        print('  Atasco en ' + columna + ' idx=' + str(start_idx) + ' dur=' + str(duracion) + ' val=' + str(round(stuck_value, 2)))
        secuencias_creadas += 1

    if secuencias_creadas < num_secuencias_stuck:
        print('Aviso: creadas ' + str(secuencias_creadas) + ' de ' + str(num_secuencias_stuck) + ' tras ' + str(intentos) + ' intentos.')
    print('Inyeccion completada. Secuencias creadas: ' + str(secuencias_creadas))
    return df_modificado

In [ ]:
if 'df_trabajo' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    num_secuencias = 50
    duracion_min = 5     # Minimo 5 periodos de 1 minuto (5 minutos)
    duracion_max = 20    # Maximo 20 periodos de 1 minuto (20 minutos)

    etiquetas_deteccion_antes_stuck = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_stuck = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print('Iniciando inyeccion de ' + str(num_secuencias) + ' secuencias de Sensor Atascado...')
    df_trabajo = inyectar_sensor_atascado(df_trabajo,
                                        num_secuencias,
                                        duracion_min,
                                        duracion_max,
                                        columnas_objetivo_existentes,
                                        random_state=42)

    print('--- Resultados de la Inyeccion de Sensor Atascado ---')
    print('Conteo etiqueta_deteccion ANTES:')
    print(etiquetas_deteccion_antes_stuck)
    print('Conteo etiqueta_deteccion DESPUES:')
    print(df_trabajo['etiqueta_deteccion'].value_counts())
    print('Conteo etiqueta_tipo_anomalia ANTES:')
    print(etiquetas_tipo_antes_stuck)
    print('Conteo etiqueta_tipo_anomalia DESPUES:')
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    num_atascados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Sensor Atascado').sum()
    print('Filas etiquetadas como Sensor Atascado: ' + str(num_atascados))
else:
    print('Error: df_trabajo o columnas_objetivo_existentes no estan definidos.')
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())

## 4. Ruido / Picos Aleatorios
Picos de 3 a 5 desviaciones estándar por encima o por debajo del valor normal.

In [ ]:
# Reutilizamos la lista definida anteriormente
if 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print(f"Se considerarán las siguientes columnas para simular 'Ruido / Picos': \n{columnas_objetivo_existentes}")
else:
    # Esta definición es por si se ejecuta esta celda de forma aislada.
    columnas_sensores_y_actuadores = [ 
        'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
        'XCO2I', 'XHINV', 'XTINV', 'XTS',
        'UVENT_cen', 'UVENT_lN'
    ]
    columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]
    if not columnas_objetivo_existentes:
        print("ADVERTENCIA: 'columnas_objetivo_existentes' está vacía o no definida.")
    else:
        print(f"Se utilizarán las siguientes columnas para simular 'Ruido / Picos': \n{columnas_objetivo_existentes}")

# Calcular la desviación estándar de las columnas objetivo a partir del DataFrame ORIGINAL para tener una medida de variabilidad 'limpia'.
std_devs_originales = {}
if 'df_original' in locals() and columnas_objetivo_existentes:
    print("\nCalculando desviaciones estándar de las columnas objetivo (desde df_original):")
    for col in columnas_objetivo_existentes:
        # Asegurarse de que la columna sea numérica antes de calcular std
        if pd.api.types.is_numeric_dtype(df_original[col]):
            std_devs_originales[col] = df_original[col].std()
            # print(f"  Std de {col}: {std_devs_originales[col]:.3f}")
        else:
            print(f"  Advertencia: Columna '{col}' no es numérica en df_original. No se calculará std.")
            std_devs_originales[col] = 0 # O manejar de otra forma, ej: no incluirla para ruido
    if not std_devs_originales:
        print("Advertencia: No se pudieron calcular desviaciones estándar. Verifica 'df_original'.")
elif not columnas_objetivo_existentes:
    print("Advertencia: 'columnas_objetivo_existentes' no está definida. No se pueden calcular std.")
else:
    print("Advertencia: 'df_original' no está definido. No se pueden calcular std de referencia.")

In [ ]:
def inyectar_ruido_picos(df, porcentaje_filas_ruido, columnas_objetivo, std_devs_ref,
                            factor_ruido_std_min=3, factor_ruido_std_max=5, num_sensores_ruido_por_fila=1, random_state=42):
    """Inyecta ruido/picos de forma vectorizada."""
    if not columnas_objetivo or not std_devs_ref:
        print('No se proporcionaron columnas o std_devs_ref.')
        return df
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    cols_validas = [c for c in columnas_objetivo if std_devs_ref.get(c, 0) > 0]
    if not cols_validas:
        print('No hay columnas con std > 0.')
        return df_modificado
    idx_normales = df_modificado.index[df_modificado['etiqueta_deteccion'] == 'normal'].to_numpy()
    n = int(len(idx_normales) * porcentaje_filas_ruido)
    if n == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado
    filas = rng.choice(idx_normales, size=n, replace=False)
    print('Inyectando ruido/picos en ' + str(n) + ' filas...')
    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    num_cols = min(num_sensores_ruido_por_fila, len(cols_validas))
    col_indices = {c: df_modificado.columns.get_loc(c) for c in cols_validas}
    ruido_count = 0
    for fila_idx in filas:
        elegidas = rng.choice(cols_validas, size=num_cols, replace=False)
        for col in elegidas:
            val = df_modificado.iat[fila_idx, col_indices[col]]
            if pd.isna(val):
                continue
            factor = rng.uniform(factor_ruido_std_min, factor_ruido_std_max)
            signo  = rng.choice([-1, 1])
            df_modificado.iat[fila_idx, col_indices[col]] = val + signo * factor * std_devs_ref[col]
            ruido_count += 1
        df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx] = 'Ruido'
    print('Inyeccion de ruido completada. ' + str(ruido_count) + ' puntos modificados en ' + str(n) + ' filas.')
    return df_modificado

In [ ]:
if ('df_trabajo' in locals() and 
    'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes and
    'std_devs_originales' in locals() and std_devs_originales):

    # Parámetros para la inyección de "Ruido / Picos Aleatorios"
    porcentaje_filas_con_ruido = 0.03  # Afectar al 3% de las filas 'normales'
    factor_std_min = 3.0               # Ruido será al menos 3 * std
    factor_std_max = 6.0               # Ruido será como máximo 6 * std
    num_sensores_afectados_por_fila = 1 # Inyectar ruido en 1 sensor por fila afectada

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    etiquetas_deteccion_antes_ruido = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_ruido = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Ruido / Picos Aleatorios'...")
    df_trabajo = inyectar_ruido_picos(df_trabajo,
                                        porcentaje_filas_con_ruido,
                                        columnas_objetivo_existentes,
                                        std_devs_originales,
                                        factor_std_min,
                                        factor_std_max,
                                        num_sensores_afectados_por_fila)

    print("\n--- Resultados de la Inyección de 'Ruido / Picos Aleatorios' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_ruido)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_ruido)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    num_ruido_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Ruido').sum()
    print(f"\nNúmero total de filas etiquetadas como 'Ruido': {num_ruido_etiquetados}")

else:
    print("Error: Asegúrate de que 'df_trabajo', 'columnas_objetivo_existentes' y 'std_devs_originales' estén definidos.")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())

## 5. Valores Fuera de Rango
**v2:** los límites se calculan como percentiles P0.1 y P99.9 del propio dataset (estadístico).
Esto difiere de v3, que usa rangos físicos reales del hardware (M3).

In [ ]:
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print("Calculando estadísticas de rango para las columnas objetivo (usando df_original):\n")
    
    stats_list = []

    for col in columnas_objetivo_existentes:
        if pd.api.types.is_numeric_dtype(df_original[col]):
            # Calcular estadísticas
            min_obs = df_original[col].min()
            max_obs = df_original[col].max()
            mean_obs = df_original[col].mean()
            std_obs = df_original[col].std()
            
            # Calcular percentiles
            p0_1 = df_original[col].quantile(0.001) # 0.1-th percentile
            p1 = df_original[col].quantile(0.01)   # 1st percentile
            p5 = df_original[col].quantile(0.05)   # 5th percentile
            p95 = df_original[col].quantile(0.95)  # 95th percentile
            p99 = df_original[col].quantile(0.99)  # 99th percentile
            p99_9 = df_original[col].quantile(0.999)# 99.9-th percentile
            
            stats_list.append({
                'Columna': col,
                'Min Observado': min_obs,
                'P0.1 (0.1%)': p0_1,
                'P1 (1%)': p1,
                'P5 (5%)': p5,
                'Media': mean_obs,
                'P95 (95%)': p95,
                'P99 (99%)': p99,
                'P99.9 (99.9%)': p99_9,
                'Max Observado': max_obs,
                'Std Dev': std_obs
            })
        else:
            print(f"  Advertencia: Columna '{col}' no es numérica. Se omitirán las estadísticas.")

    if stats_list:
        df_stats = pd.DataFrame(stats_list)
        print(df_stats.to_string()) # .to_string() para mostrar todas las filas y columnas
    else:
        print("No se pudieron calcular estadísticas para ninguna columna.")

else:
    print("Error: 'df_original' o 'columnas_objetivo_existentes' no están definidos o la lista de columnas está vacía.")

In [ ]:
# Verificamos que las variables necesarias existan
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals():
    print("=== CALCULANDO LÍMITES AUTOMÁTICOS (P0.1 y P99.9) ===\n")
    
    stats_list = []
    # Este diccionario guardará los límites para usarlos después en el modelo
    limites_automaticos = {} 

    for col in columnas_objetivo_existentes:
        if pd.api.types.is_numeric_dtype(df_original[col]):
            # 1. Calcular Percentiles Extremos
            p0_1 = df_original[col].quantile(0.001)  # Límite Inferior Automático
            p99_9 = df_original[col].quantile(0.999) # Límite Superior Automático
            
            # 2. Calcular otras estadísticas para el reporte
            min_obs = df_original[col].min()
            max_obs = df_original[col].max()
            mean_obs = df_original[col].mean()
            
            # 3. Guardar en el diccionario de límites
            limites_automaticos[col] = {
                'min': p0_1,
                'max': p99_9
            }
            
            # 4. Agregar a la lista para mostrar la tabla resumen
            stats_list.append({
                'Columna': col,
                'Min Absoluto': min_obs,
                'Límite Inf (P0.1)': p0_1,
                'Media': mean_obs,
                'Límite Sup (P99.9)': p99_9,
                'Max Absoluto': max_obs
            })
        else:
            print(f"⚠️ Advertencia: '{col}' no es numérica. Omitida.")

    # Mostrar la tabla de resultados
    if stats_list:
        df_stats = pd.DataFrame(stats_list)
        print(df_stats.to_string(index=False))
        print("\n✅ Límites definidos exitosamente en la variable 'limites_automaticos'.")
        
        rangos_validos_sensores = limites_automaticos
    else:
        print("❌ No se calcularon estadísticas.")

else:
    print("❌ Error: df_original o columnas_objetivo_existentes no definidos.")

# --- EJEMPLO DE CÓMO SE USARÍAN ESTOS VALORES AUTOMÁTICOS ---
# Si quisieras ver el límite de una variable:
# print(f"El límite superior de XTS es: {limites_automaticos['XTS']['max']}")
# En v2 los rangos son estadísticos (P0.1/P99.9), no físicos como en v3
limites_automaticos_v2 = limites_automaticos if 'limites_automaticos' in locals() else {}
rangos_validos_sensores = limites_automaticos_v2

In [ ]:
def inyectar_valores_fuera_rango(df, porcentaje_filas_afectadas, rangos_sensores,
                                    columnas_con_rango, num_sensores_afectados_por_fila=1,
                                    margen_error_factor=0.1, random_state=42):
    """Inyecta valores fuera de rango de forma vectorizada."""
    if not columnas_con_rango or not rangos_sensores:
        print('No se proporcionaron columnas o rangos.')
        return df
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    idx_normales = df_modificado.index[df_modificado['etiqueta_deteccion'] == 'normal'].to_numpy()
    n = int(len(idx_normales) * porcentaje_filas_afectadas)
    if n == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado
    filas = rng.choice(idx_normales, size=n, replace=False)
    print('Inyectando valores fuera de rango en ' + str(n) + ' filas...')
    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    col_indices = {c: df_modificado.columns.get_loc(c) for c in columnas_con_rango}
    count = 0
    for fila_idx in filas:
        elegidas = rng.choice(columnas_con_rango, size=min(num_sensores_afectados_por_fila, len(columnas_con_rango)), replace=False)
        for col in elegidas:
            if col not in rangos_sensores:
                continue
            val = df_modificado.iat[fila_idx, col_indices[col]]
            if pd.isna(val):
                continue
            r = rangos_sensores[col]
            span = r['max'] - r['min']
            margen = span * margen_error_factor if span > 0 else abs(r['max']) * 0.1 or 1.0
            nuevo = (r['min'] - margen) if rng.random() < 0.5 else (r['max'] + margen)
            df_modificado.iat[fila_idx, col_indices[col]] = nuevo
            count += 1
        df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx] = 'Valores Fuera de Rango'
    print('Inyeccion fuera de rango completada. ' + str(count) + ' puntos en ' + str(n) + ' filas.')
    return df_modificado

In [ ]:
# 1. Recuperamos los límites automáticos
if 'limites_automaticos' in locals():
    rangos_validos_sensores = limites_automaticos

# 2. Generamos la lista de columnas basada en el diccionario
if 'rangos_validos_sensores' in locals():
    columnas_con_rango_definido = list(rangos_validos_sensores.keys())

# 3. Aseguramos que df_trabajo exista
if 'df_trabajo' not in locals() and 'df_original' in locals():
    df_trabajo = df_original.copy()
    print("✅ 'df_trabajo' creado a partir de 'df_original'.")

# --- FIN DE LA ADAPTACIÓN ---


if ('df_trabajo' in locals() and 
    'columnas_con_rango_definido' in locals() and columnas_con_rango_definido and
    'rangos_validos_sensores' in locals() and rangos_validos_sensores):

    # Parámetros para la inyección de "Valores Fuera de Rango"
    porcentaje_filas_oor = 0.02  # Afectar al 2% de las filas 'normales'
    margen_factor = 0.1          # Exceder el límite por un 10% del span del rango válido
    num_sensores_afectados = 1   # Inyectar en 1 sensor por fila afectada

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    # (Añadido manejo de error por si las columnas no existen aún)
    if 'etiqueta_deteccion' not in df_trabajo.columns:
        df_trabajo['etiqueta_deteccion'] = 'normal'
    if 'etiqueta_tipo_anomalia' not in df_trabajo.columns:
        df_trabajo['etiqueta_tipo_anomalia'] = 'Ninguna'

    etiquetas_deteccion_antes_oor = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_oor = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Valores Fuera de Rango'...")
    df_trabajo = inyectar_valores_fuera_rango(df_trabajo,
                                                porcentaje_filas_oor,
                                                rangos_validos_sensores,
                                                columnas_con_rango_definido,
                                                num_sensores_afectados,
                                                margen_factor)

    print("\n--- Resultados de la Inyección de 'Valores Fuera de Rango' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_oor)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_oor)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    num_oor_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Valores Fuera de Rango').sum()
    print(f"\nNúmero total de filas etiquetadas como 'Valores Fuera de Rango': {num_oor_etiquetados}")

else:
    print("Error: Asegúrate de que 'df_trabajo', 'columnas_con_rango_definido' y 'rangos_validos_sensores' estén definidos y no vacíos.")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())

## 6. Desviación de Correlación
Rompe la relación entre pares de sensores correlacionados. v2 incluye el par (PRAD, UVENT_cen) que en v3 fue sustituido por (PTEXT, XTINV).

In [ ]:
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print("Calculando la matriz de correlación para las columnas objetivo (usando df_original):\n")
    
    # Asegurarse de que solo se usen columnas numéricas para la correlación
    columnas_numericas_para_corr = df_original[columnas_objetivo_existentes].select_dtypes(include=np.number).columns
    
    if len(columnas_numericas_para_corr) < 2:
        print("No hay suficientes columnas numéricas para calcular una matriz de correlación significativa.")
    else:
        correlation_matrix = df_original[columnas_numericas_para_corr].corr()

        print("Matriz de Correlación:")
        print(correlation_matrix)

        # Visualizar la matriz de correlación usando un heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
        plt.title('Matriz de Correlación de Sensores y Actuadores (sobre df_original)')
        plt.show()
else:
    print("Error: 'df_original' o 'columnas_objetivo_existentes' no están definidos o la lista de columnas está vacía.")

In [ ]:
# Columnas para las que podríamos necesitar percentiles (basadas en los pares sugeridos)
columnas_para_percentiles = list(set(['PRAD', 'PRGINT', 'XTINV', 'XHINV', 'UVENT_cen']))
percentiles_calculados = {}

if 'df_original' in locals():
    print("Calculando percentiles para columnas seleccionadas (desde df_original):")
    for col in columnas_para_percentiles:
        if col in df_original.columns and pd.api.types.is_numeric_dtype(df_original[col]):
            p25 = df_original[col].quantile(0.25)
            p50 = df_original[col].quantile(0.50) # Mediana
            p75 = df_original[col].quantile(0.75)
            percentiles_calculados[col] = {'p25': p25, 'p50': p50, 'p75': p75}
            # print(f"  {col}: P25={p25:.2f}, P50={p50:.2f}, P75={p75:.2f}")
        else:
            print(f"  Advertencia: Columna '{col}' no es numérica o no existe. Se omitirán sus percentiles.")
else:
    print("Error: 'df_original' no está definido. No se pueden calcular percentiles.")
# v2: incluye UVENT_cen en los percentiles

In [ ]:
def inyectar_desviacion_correlacion(df, lista_pares, percentiles_ref, porcentaje_filas_afectadas,
                                    filtro_hora_inicio=None, filtro_hora_fin=None, random_state=42):
    """Inyecta desviacion de correlacion. Filtro temporal vectorizado (sin apply fila a fila)."""
    if not lista_pares or not percentiles_ref:
        print('Lista de pares o percentiles no proporcionados.')
        return df
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    mask_normal = df_modificado['etiqueta_deteccion'] == 'normal'
    if filtro_hora_inicio is not None and filtro_hora_fin is not None:
        if filtro_hora_inicio <= filtro_hora_fin:
            mask_hora = (df_modificado['Hora'] >= filtro_hora_inicio) & (df_modificado['Hora'] <= filtro_hora_fin)
        else:
            mask_hora = (df_modificado['Hora'] >= filtro_hora_inicio) | (df_modificado['Hora'] <= filtro_hora_fin)
        mask_normal = mask_normal & mask_hora
    idx_candidatos = df_modificado.index[mask_normal].to_numpy()
    n = int(len(idx_candidatos) * porcentaje_filas_afectadas)
    if n == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado
    filas = rng.choice(idx_candidatos, size=n, replace=False)
    print('Inyectando desviacion de correlacion en ' + str(n) + ' filas...')
    det_idx  = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    count = 0
    pares = list(lista_pares)
    for fila_idx in filas:
        sA, sB = pares[int(rng.integers(0, len(pares)))]
        if sA not in percentiles_ref or sB not in percentiles_ref:
            continue
        vA = df_modificado.at[fila_idx, sA]
        vB = df_modificado.at[fila_idx, sB]
        if pd.isna(vA) or pd.isna(vB):
            continue
        corr = correlation_matrix.loc[sA, sB]
        pB = percentiles_ref[sB]
        pA = percentiles_ref[sA]
        nuevo = None
        if vB > pB['p75']:
            nuevo = pA['p25'] if corr > 0.3 else (pA['p75'] if corr < -0.3 else None)
        elif vB < pB['p25']:
            nuevo = pA['p75'] if corr > 0.3 else (pA['p25'] if corr < -0.3 else None)
        if nuevo is None:
            continue
        df_modificado.at[fila_idx, sA] = nuevo
        df_modificado.iat[fila_idx, det_idx]  = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx] = 'Desviacion de Correlacion'
        count += 1
    print('Inyeccion de correlacion completada. ' + str(count) + ' puntos modificados.')
    return df_modificado

In [ ]:
if ('df_trabajo' in locals() and
    'percentiles_calculados' in locals() and percentiles_calculados and
    'correlation_matrix' in locals()):

    pares_para_desviacion = [
        ('PRAD', 'PRGINT'),
        ('XTINV', 'XHINV'),
        ('PRAD', 'UVENT_cen')
    ]
    porcentaje_filas_desviacion = 0.03

    etiquetas_deteccion_antes_corr = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_corr = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print('Iniciando inyeccion de Desviacion de Correlacion...')
    df_trabajo = inyectar_desviacion_correlacion(df_trabajo,
                                                pares_para_desviacion,
                                                percentiles_calculados,
                                                porcentaje_filas_desviacion,
                                                filtro_hora_inicio=7,
                                                filtro_hora_fin=18,
                                                random_state=42)

    print('Conteo etiqueta_deteccion ANTES:')
    print(etiquetas_deteccion_antes_corr)
    print('Conteo etiqueta_deteccion DESPUES:')
    print(df_trabajo['etiqueta_deteccion'].value_counts())
    print('Conteo etiqueta_tipo_anomalia ANTES:')
    print(etiquetas_tipo_antes_corr)
    print('Conteo etiqueta_tipo_anomalia DESPUES:')
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print('Error: df_trabajo, percentiles_calculados o correlation_matrix no definidos.')
# Nota: en v2 pares_para_desviacion incluye ('PRAD', 'UVENT_cen')
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())

## 7. Anomalías Contextuales
### 7a. Luz Interior Anómala Nocturna
Radiación interior alta en horas nocturnas — fallo de sensor o actuador.

In [ ]:
def inyectar_luz_nocturna_anomala(df, porcentaje_filas_afectadas,
                                    hora_noche_inicio=23, hora_noche_fin=4,
                                    prad_umbral_cero=1.0,
                                    prgint_anomalo_min=20.0, prgint_anomalo_max=50.0, random_state=42):
    """Inyecta luz nocturna anomala de forma vectorizada."""
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    if hora_noche_inicio <= hora_noche_fin:
        es_noche = (df_modificado['Hora'] >= hora_noche_inicio) & (df_modificado['Hora'] <= hora_noche_fin)
    else:
        es_noche = (df_modificado['Hora'] >= hora_noche_inicio) | (df_modificado['Hora'] <= hora_noche_fin)
    mask = es_noche & (df_modificado['PRAD'] < prad_umbral_cero) & (df_modificado['etiqueta_deteccion'] == 'normal')
    idx_candidatos = df_modificado.index[mask].to_numpy()
    if len(idx_candidatos) == 0:
        print('No se encontraron filas candidatas para luz nocturna anomala.')
        return df_modificado
    n = int(len(idx_candidatos) * porcentaje_filas_afectadas)
    if n == 0:
        print('Resultado 0 filas a afectar.')
        return df_modificado
    filas = rng.choice(idx_candidatos, size=n, replace=False)
    print('Inyectando luz nocturna anomala en ' + str(n) + ' filas...')
    prgint_idx = df_modificado.columns.get_loc('PRGINT')
    det_idx    = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx   = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    valores = rng.uniform(prgint_anomalo_min, prgint_anomalo_max, size=n)
    for i, fila_idx in enumerate(filas):
        df_modificado.iat[fila_idx, prgint_idx] = valores[i]
        df_modificado.iat[fila_idx, det_idx]    = 'anomalia'
        df_modificado.iat[fila_idx, tipo_idx]   = 'Contextual'
    print('Inyeccion de luz nocturna completada. ' + str(n) + ' filas modificadas.')
    return df_modificado

# --- Uso de la función para Luz Nocturna Anómala ---
if 'df_trabajo' in locals():
    porcentaje_luz_nocturna = 0.01 # Afectar al 1% de las filas candidatas

    # Guardar el estado de las etiquetas antes
    etiquetas_deteccion_antes_ln = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_ln = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Luz Nocturna Anómala'...")
    df_trabajo = inyectar_luz_nocturna_anomala(df_trabajo, porcentaje_luz_nocturna)

    print("\n--- Resultados de la Inyección de 'Luz Nocturna Anómala' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_ln)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_ln)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print("Error: 'df_trabajo' no está definido.")

In [ ]:
if 'df_trabajo' in locals():
    cfg = INYECCION['contextual_luz']
    df_trabajo = inyectar_luz_nocturna_anomala(
        df_trabajo,
        porcentaje_filas_afectadas=cfg['porcentaje_filas'],
        hora_noche_inicio=cfg['hora_noche_inicio'],
        hora_noche_fin=cfg['hora_noche_fin'],
        prad_umbral_cero=cfg['prad_umbral_cero'],
        prgint_anomalo_min=cfg['prgint_min'],
        prgint_anomalo_max=cfg['prgint_max'],
    )
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())


### 7b. CO2 Sin Respuesta a Ventilación
**v2:** condición estricta — ventanas abiertas >70% Y gradiente >50 ppm, duración 5-15 min.
*(v3 relaja esto a 30 ppm y 3-8 min — mejora M9)*

In [ ]:
def inyectar_co2_sin_respuesta_ventilacion(df, num_secuencias_afectadas,
                                            umbral_vent_abierta=70.0,
                                            umbral_diff_co2=50.0,
                                            duracion_anomalia_min=2, duracion_anomalia_max=4,
                                            random_state=42):
    """
    Inyecta anomalias de CO2 sin respuesta a ventilacion.
    Vectorizado con cumsum: busca segmentos validos en O(N) en vez de O(N x intentos).
    """
    rng = np.random.default_rng(random_state)
    df_modificado = df.copy()
    n = len(df_modificado)
    es_normal = (df_modificado['etiqueta_deteccion'] == 'normal').values.copy()
    vent_cen  = df_modificado['UVENT_cen'].values
    vent_ln   = df_modificado['UVENT_lN'].values
    co2ext    = df_modificado['PCO2EXT'].values
    co2int    = df_modificado['XCO2I'].values
    xco2i_idx = df_modificado.columns.get_loc('XCO2I')
    det_idx   = df_modificado.columns.get_loc('etiqueta_deteccion')
    tipo_idx  = df_modificado.columns.get_loc('etiqueta_tipo_anomalia')
    secuencias_creadas = 0
    max_intentos = num_secuencias_afectadas * 10
    print('Intentando crear ' + str(num_secuencias_afectadas) + ' secuencias de CO2 sin respuesta a ventilacion...')
    intentos = 0
    while secuencias_creadas < num_secuencias_afectadas and intentos < max_intentos:
        intentos += 1
        duracion = int(rng.integers(duracion_anomalia_min, duracion_anomalia_max + 1))
        max_start = n - duracion
        if max_start <= 0:
            continue
        cond_contexto = (
            ((vent_cen[:max_start] > umbral_vent_abierta) | (vent_ln[:max_start] > umbral_vent_abierta)) &
            (co2ext[:max_start] < co2int[:max_start] - umbral_diff_co2) &
            ~np.isnan(co2int[:max_start])
        )
        es_valido = es_normal & ~np.isnan(co2int)
        cum = np.cumsum(es_valido.astype(np.int32))
        cum = np.concatenate([[0], cum])
        sumas = cum[duracion: max_start + duracion] - cum[:max_start]
        candidatos = np.where(cond_contexto & (sumas == duracion))[0]
        if len(candidatos) == 0:
            continue
        start_idx = int(rng.choice(candidatos))
        xco2i_base = float(co2int[start_idx])
        print('  CO2 sin respuesta idx=' + str(start_idx) + ' dur=' + str(duracion))
        for k in range(duracion):
            ci = start_idx + k
            df_modificado.iat[ci, xco2i_idx] = xco2i_base + float(rng.uniform(-2, 5))
            df_modificado.iat[ci, det_idx]   = 'anomalia'
            df_modificado.iat[ci, tipo_idx]  = 'Contextual'
        es_normal[start_idx:start_idx + duracion] = False
        secuencias_creadas += 1
    if secuencias_creadas < num_secuencias_afectadas:
        print('Aviso: creadas ' + str(secuencias_creadas) + ' de ' + str(num_secuencias_afectadas) + ' tras ' + str(intentos) + ' intentos.')
    print('Inyeccion CO2 completada. Secuencias: ' + str(secuencias_creadas))
    return df_modificado

# --- Uso de la función para CO2 sin Respuesta a Ventilación ---
if 'df_trabajo' in locals():
    num_secuencias_co2_contextual = 33 # Intentar crear 100 secuencias de este tipo

    # Guardar el estado de las etiquetas antes
    etiquetas_deteccion_antes_co2 = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_co2 = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'CO2 sin respuesta a ventilación'...")
    df_trabajo = inyectar_co2_sin_respuesta_ventilacion(df_trabajo, num_secuencias_co2_contextual)

    print("\n--- Resultados de la Inyección de 'CO2 sin respuesta a ventilación' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_co2)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_co2)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print("Error: 'df_trabajo' no está definido.")

In [ ]:
if 'df_trabajo' in locals():
    cfg = INYECCION['contextual_co2']
    df_trabajo = inyectar_co2_sin_respuesta_ventilacion(
        df_trabajo,
        num_secuencias_afectadas=cfg['num_secuencias'],
        umbral_vent_abierta=cfg['umbral_vent_abierta'],
        umbral_diff_co2=cfg['umbral_diff_co2'],
        duracion_anomalia_min=cfg['duracion_min'],
        duracion_anomalia_max=cfg['duracion_max'],
    )
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())


## 8. Resumen de etiquetas inyectadas

In [ ]:
if 'df_trabajo' in locals():
    print("Conteo completo de valores en 'etiqueta_tipo_anomalia':")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))

    print("\nConteo completo de valores en 'etiqueta_deteccion':")
    print(df_trabajo['etiqueta_deteccion'].value_counts(dropna=False))
    
    # Un pequeño resumen del total de filas y filas anómalas
    total_filas = len(df_trabajo)
    filas_anomalas_detectadas = (df_trabajo['etiqueta_deteccion'] == 'anomalia').sum()
    print(f"\nTotal de filas en df_trabajo: {total_filas}")
    print(f"Total de filas marcadas como 'anomalia' en 'etiqueta_deteccion': {filas_anomalas_detectadas}")
    if total_filas > 0:
        print(f"Porcentaje de filas anómalas: {filas_anomalas_detectadas / total_filas * 100:.2f}%")

else:
    print("Error: 'df_trabajo' no está definido.")

## 9. Visualización rápida

In [ ]:
# --- Visualización ---
columna = "XTINV" # Variable a visualizar

# Asegurarse de que los dataframes existan y la columna 'Fecha' sea el índice
# (Esta parte podría ser necesaria si los DataFrames en memoria no tienen el índice configurado)
try:
    if 'df_original' in locals():
        if 'Fecha' in df_original.columns:
            df_original['Fecha'] = pd.to_datetime(df_original['Fecha'])
            df_original.set_index('Fecha', inplace=True)
        elif not isinstance(df_original.index, pd.DatetimeIndex):
            print("Advertencia: df_original está en memoria pero no tiene un DatetimeIndex ni columna 'Fecha' adecuada.")
            # Decide cómo manejar esto, quizás mostrar un error o no graficar df_original

    if 'df_trabajo' in locals():
        if 'Fecha' in df_trabajo.columns:
            df_trabajo['Fecha'] = pd.to_datetime(df_trabajo['Fecha'])
            df_trabajo.set_index('Fecha', inplace=True)
        elif not isinstance(df_trabajo.index, pd.DatetimeIndex):
                print("Advertencia: df_trabajo está en memoria pero no tiene un DatetimeIndex ni columna 'Fecha' adecuada.")
            # Decide cómo manejar esto

except Exception as e:
    print(f"Error al configurar índices para DataFrames en memoria: {e}")


if 'df_original' in locals() and 'df_trabajo' in locals():
    
    fig, axes = plt.subplots(2, 1, figsize=(22, 10), sharex=True) # Ejemplo

    # Graficar df_original
    if 'df_original' in locals() and hasattr(df_original, 'index') and isinstance(df_original.index, pd.DatetimeIndex) and columna in df_original.columns:
        axes[0].plot(df_original.index, df_original[columna], label="Original (df_original - en memoria)", alpha=0.7, color="blue")
        axes[0].set_title(f"{columna} - Datos Originales (df_original - en memoria)")
        axes[0].set_ylabel(columna)
        axes[0].legend()
        axes[0].grid(True, linestyle='--', alpha=0.5)
    elif 'df_original' in locals():
            axes[0].set_title(f"{columna} - Datos Originales (df_original en memoria, pero '{columna}' no encontrada o índice incorrecto)")
    else:
        axes[0].set_title(f"df_original no encontrado en memoria")

    # Graficar df_trabajo (similar lógica de verificación)
    if 'df_trabajo' in locals() and hasattr(df_trabajo, 'index') and isinstance(df_trabajo.index, pd.DatetimeIndex) and columna in df_trabajo.columns and 'etiqueta_tipo_anomalia' in df_trabajo.columns:
        axes[1].plot(df_trabajo.index, df_trabajo[columna], label="Con Anomalías (df_trabajo - en memoria)", linestyle="dashed", alpha=0.7, color="orange")
        
        indices_con_anomalias_inyectadas = df_trabajo[df_trabajo['etiqueta_tipo_anomalia'].notna()].index
        if not indices_con_anomalias_inyectadas.empty:
            axes[1].scatter(
                df_trabajo.loc[indices_con_anomalias_inyectadas].index,
                df_trabajo.loc[indices_con_anomalias_inyectadas, columna],
                color='red', label='Anomalías inyectadas', s=50, edgecolors='black', zorder=3
            )
        axes[1].set_title(f"{columna} - Datos con Anomalías (df_trabajo - en memoria)")
        axes[1].set_xlabel("Fecha")
        axes[1].set_ylabel(columna)
        axes[1].legend()
        axes[1].grid(True, linestyle='--', alpha=0.5)
    elif 'df_trabajo' in locals():
        axes[1].set_title(f"{columna} - Datos con Anomalías (df_trabajo en memoria, pero '{columna}' o 'etiqueta_tipo_anomalia' no encontrada o índice incorrecto)")
    else:
        axes[1].set_title(f"df_trabajo no encontrado en memoria")

    plt.tight_layout()
    plt.show()
else:
    print("Los DataFrames df_original y/o df_trabajo no están disponibles en memoria. No se puede generar la visualización.")

## 10. Guardar resultado

In [ ]:
# Restaurar Fecha como columna si la visualización la convirtió a índice
if isinstance(df_trabajo.index, pd.DatetimeIndex) and df_trabajo.index.name == 'Fecha':
    df_trabajo = df_trabajo.reset_index()
df_trabajo.to_parquet(PARQUET_02, index=False)
print(f"Guardado: {PARQUET_02}")
print(f"Shape: {df_trabajo.shape}")
print("\nDistribución final de etiquetas:")
print(df_trabajo['etiqueta_deteccion'].value_counts())
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts())
